# PyTorch 基础教程

本笔记包含 PyTorch 的基础概念和实践，包括：
- 张量操作
- 自动微分
- 神经网络基础
- 数据可视化

In [4]:
# 导入必要的库
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# 设置随机种子以确保结果可重现（伪随机）
torch.manual_seed(42)
np.random.seed(42)

In [5]:
# GPU测试代码 - RTX 5070 兼容性修复
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import torch
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 版本: {torch.version.cuda}")
print(f"GPU 可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"显卡名称: {torch.cuda.get_device_name(0)}")
    print(f"显卡内存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    
    capability = torch.cuda.get_device_capability(0)
    print(f"GPU计算能力: {capability[0]}.{capability[1]}")
    
    # RTX 5070 计算能力12.0需要更新的PyTorch版本
    if capability[0] >= 12:
        print(f"⚠️  检测到计算能力{capability[0]}.{capability[1]}的新GPU")
        print("当前PyTorch版本可能不完全支持，建议升级到最新nightly版本")
    
    try:
        torch.cuda.empty_cache()
        print("\n开始GPU兼容性测试...")
        
        # 设置CUDA上下文
        torch.cuda.synchronize()
        
        # 使用更兼容的操作
        x = torch.ones(1, dtype=torch.float32, device='cuda')
        print("✓ 基础GPU张量创建成功")
        
        y = x + 1
        torch.cuda.synchronize()
        print("✓ 基础GPU计算成功")
        
        # 小矩阵测试
        A = torch.ones(5, 5, dtype=torch.float32, device='cuda')
        B = A + A
        torch.cuda.synchronize()
        print("✓ GPU矩阵运算成功")
        
        device = torch.device('cuda')
        print(f"✓ GPU设备可用: {device}")
        
        del x, y, A, B
        torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"GPU测试失败: {e}")
        device = torch.device('cpu')
        print(f"回退到CPU模式")

else:
    device = torch.device('cpu')
    print(f"GPU不可用，使用CPU")

print(f"\n当前工作设备: {device}")

PyTorch 版本: 2.8.0+cu128
CUDA 版本: 12.8
GPU 可用: True
显卡名称: NVIDIA GeForce RTX 5070
显卡内存: 11.49 GB
GPU计算能力: 12.0
⚠️  检测到计算能力12.0的新GPU
当前PyTorch版本可能不完全支持，建议升级到最新nightly版本

开始GPU兼容性测试...
✓ 基础GPU张量创建成功
✓ 基础GPU计算成功
✓ GPU矩阵运算成功
✓ GPU设备可用: cuda

当前工作设备: cuda


## 张量基础操作

In [6]:
# 创建基本张量
print("=== 张量创建 ===")
x = torch.rand(3, 4, device=device)  # 使用之前定义的device
print(f"随机张量 (3x4):\n{x}")
print(f"张量形状: {x.shape}")
print(f"张量数据类型: {x.dtype}")
print(f"张量设备: {x.device}")

# 也可以这样创建GPU张量
x_gpu = torch.rand(3, 4, device='cuda')
print(f"\nGPU张量:\n{x_gpu}")
print(f"GPU张量设备: {x_gpu.device}")

# 或者先创建再移动到GPU
x_cpu = torch.rand(3, 4)
x_moved = x_cpu.to(device)
print(f"\n移动到GPU的张量设备: {x_moved.device}")

=== 张量创建 ===
随机张量 (3x4):
tensor([[0.6130, 0.0101, 0.3984, 0.0403],
        [0.1563, 0.4825, 0.7362, 0.4060],
        [0.5189, 0.2867, 0.2416, 0.9228]], device='cuda:0')
张量形状: torch.Size([3, 4])
张量数据类型: torch.float32
张量设备: cuda:0

GPU张量:
tensor([[0.9877, 0.1289, 0.5621, 0.5221],
        [0.7445, 0.5955, 0.9647, 0.8979],
        [0.7730, 0.6681, 0.5462, 0.5071]], device='cuda:0')
GPU张量设备: cuda:0

移动到GPU的张量设备: cuda:0


In [7]:
# 创建不同类型的张量
print("=== 不同类型的张量 ===")

# 零张量和全1张量
zeros = torch.zeros(3, 4)
ones = torch.ones(2, 3)
print(f"零张量 (3x4):\n{zeros}")
print(f"全1张量 (2x3):\n{ones}")

# 范围和线性空间张量
range_tensor = torch.arange(0, 10, 2)  # 步长为2
linspace = torch.linspace(0, 10, 6)    # 6个等间距点
print(f"范围张量 (0到10，步长2): {range_tensor}")
print(f"线性空间张量 (0到10，6个点): {linspace}")

# 正态分布和均匀分布
normal = torch.randn(3, 3)  # 标准正态分布
uniform = torch.rand(3, 3)  # [0,1)均匀分布
print(f"正态分布张量:\n{normal}")
print(f"均匀分布张量:\n{uniform}")

# 从numpy数组创建张量
np_array = np.array([[1, 2, 3], [4, 5, 6]])
tensor_from_numpy = torch.from_numpy(np_array)
print(f"从numpy创建的张量:\n{tensor_from_numpy}")

=== 不同类型的张量 ===
零张量 (3x4):
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
全1张量 (2x3):
tensor([[1., 1., 1.],
        [1., 1., 1.]])
范围张量 (0到10，步长2): tensor([0, 2, 4, 6, 8])
线性空间张量 (0到10，6个点): tensor([ 0.,  2.,  4.,  6.,  8., 10.])
正态分布张量:
tensor([[ 2.2082, -0.6380,  0.4617],
        [ 0.2674,  0.5349,  0.8094],
        [ 1.1103, -1.6898, -0.9890]])
均匀分布张量:
tensor([[0.8860, 0.5832, 0.3376],
        [0.8090, 0.5779, 0.9040],
        [0.5547, 0.3423, 0.6343]])
从numpy创建的张量:
tensor([[1, 2, 3],
        [4, 5, 6]])


## 张量运算和操作

In [8]:
# 张量运算
print("=== 张量运算 ===")
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(f"张量a: {a}")
print(f"张量b: {b}")
print(f"加法 a + b: {a + b}")
print(f"减法 a - b: {a - b}")
print(f"逐元素乘法 a * b: {a * b}")
print(f"除法 a / b: {a / b}")
print(f"点积 a @ b: {a @ b}")
print(f"余弦相似度: {torch.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0))}")

# 矩阵运算
print("\n=== 矩阵运算 ===")
A = torch.randn(3, 4)
B = torch.randn(4, 2)
C = torch.mm(A, B)  # 矩阵乘法
print(f"矩阵A形状: {A.shape}")
print(f"矩阵B形状: {B.shape}")
print(f"矩阵乘法C=A@B形状: {C.shape}")

# 张量形状操作
print("\n=== 形状操作 ===")
original = torch.randn(12)
print(f"原始张量形状: {original.shape}")
reshaped = original.reshape(3, 4)
print(f"重塑为3x4: {reshaped.shape}")
transposed = reshaped.t()
print(f"转置后形状: {transposed.shape}")
flattened = reshaped.flatten()
print(f"展平后形状: {flattened.shape}")

=== 张量运算 ===
张量a: tensor([1., 2., 3.])
张量b: tensor([4., 5., 6.])
加法 a + b: tensor([5., 7., 9.])
减法 a - b: tensor([-3., -3., -3.])
逐元素乘法 a * b: tensor([ 4., 10., 18.])
除法 a / b: tensor([0.2500, 0.4000, 0.5000])
点积 a @ b: 32.0
余弦相似度: tensor([0.9746])

=== 矩阵运算 ===
矩阵A形状: torch.Size([3, 4])
矩阵B形状: torch.Size([4, 2])
矩阵乘法C=A@B形状: torch.Size([3, 2])

=== 形状操作 ===
原始张量形状: torch.Size([12])
重塑为3x4: torch.Size([3, 4])
转置后形状: torch.Size([4, 3])
展平后形状: torch.Size([12])


## 自动微分 (Autograd)

In [11]:
# 创建不同类型的张量
print("零张量:")
zeros = torch.zeros(3, 4)
print(zeros)

print("\n全1张量:")
ones = torch.ones(2, 3)
print(ones)

print("\n指定范围的张量:")
range_tensor = torch.arange(10)
print(range_tensor)

print("\n线性空间张量:")
linspace = torch.linspace(0, 10, 5)
print(linspace)

print("\n正态分布随机张量:")
normal = torch.randn(3, 3)
print(normal)

# 张量运算
print("\n基本运算:")
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])
print(f"a + b = {a + b}")
print(f"a * b = {a * b}")
print(f"a @ b = {a @ b}")  # 点积

# 张量操作
print("\n改变形状:")
c = torch.randn(12)
print(f"原始张量: {c}")
c_reshaped = c.reshape(3, 4)
print(f"重塑后: \n{c_reshaped}")

# 移动到GPU(如果可用)
if torch.cuda.is_available():
    gpu_tensor = x.to(device)
    print("\n移动到GPU的张量:", gpu_tensor.device)

零张量:
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

全1张量:
tensor([[1., 1., 1.],
        [1., 1., 1.]])

指定范围的张量:
tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

线性空间张量:
tensor([ 0.0000,  2.5000,  5.0000,  7.5000, 10.0000])

正态分布随机张量:
tensor([[ 0.5023, -1.6336,  0.7949],
        [-0.4340,  0.1795,  0.0356],
        [-0.5065, -1.2639, -0.1908]])

基本运算:
a + b = tensor([5, 7, 9])
a * b = tensor([ 4, 10, 18])
a @ b = 32

改变形状:
原始张量: tensor([-0.8826, -0.5476, -1.1466,  1.5612, -1.6720, -1.1294,  1.1089, -0.2543,
         0.5814, -0.3218, -0.4894, -0.9923])
重塑后: 
tensor([[-0.8826, -0.5476, -1.1466,  1.5612],
        [-1.6720, -1.1294,  1.1089, -0.2543],
        [ 0.5814, -0.3218, -0.4894, -0.9923]])

移动到GPU的张量: cuda:0
